# **Proyecto ML - parte 1 — Envío 4**

**Grupo 18**

Integrantes:
- Hugo Lenin Guzman Monsalve
- Jose Luis Parra Fernandez
- Julian Camilo Rivera Guzman
- Sergio Alejandro Torres Melendez

## Resumen

| Envío | Modelo | Accuracy CV (5 folds) |
| --- | --- | --- |
| 1 | TF-IDF con marcas de posición (E/P/L) por oración + Regresión Logística | 87,13 % |
| 2 | marcas E/P/L por cláusula + TF-IDF de caracteres + Regresión Logística | 89,15 % |
| 3 | misma representación + LinearSVC C=0,5 | 89,39 % |
| **4** | **representación por rol de opinión + LinearSVC C=0,5** | **90,52 %** |

Este envío cambia la **representación**, no el clasificador. El envío 3 ya había
mostrado que cambiar de Regresión Logística a LinearSVC solo daba +0,24 puntos y
que el 89 % de los errores restantes eran confusiones positivo ↔ negativo: el
límite no estaba en el clasificador sino en cómo se representaba el texto.

Resultado: **+1,13 puntos sobre el envío 3** y **+1,37 sobre el envío 2**. La clase
neutral queda prácticamente resuelta (F1 de 98,11 % a 99,68 %) y los errores totales
bajan de 1.273 a 1.137 sobre las 12.000 reseñas.

Todo el trabajo cumple los requisitos técnicos de la Parte 1: solo scikit-learn,
representaciones clásicas (bolsa de palabras, TF-IDF, n-gramas de palabras y de
caracteres), sin aprendizaje profundo, sin arquitecturas transformer y sin
embeddings preentrenados.

## 1. Diagnóstico: por qué la bolsa de palabras se estanca en 89 %

Antes de cambiar nada se midió **dónde** estaba el error del envío 3. Dos hallazgos
determinaron todo el diseño de este envío.

**Hallazgo 1: casi todo el error es positivo ↔ negativo.** La clase neutral se
clasificaba con F1 de 98,1 %, mientras que 1.136 de los 1.273 errores (89 %) eran
reseñas de opinión asignadas a la polaridad contraria.

**Hallazgo 2: las palabras de opinión no indican la clase.** Al separar el
vocabulario de opinión y contar en qué clase aparece cada palabra, resulta que está
repartido casi 50/50. Por ejemplo, `terrible` aparece en 61 reseñas **positivas** y
98 negativas; `resulto` en 1.210 positivas y 1.216 negativas. El sesgo de casi
todas las palabras es cercano a 0.

La explicación es que cada reseña mezcla a propósito opiniones de las dos
polaridades, y la etiqueta la decide **una** de ellas mientras las demás actúan como
distractores:

> Pedí esto a mediados de mes y me llegó en cuatro días. **La atención al cliente se
> queda muy corta** porque tardaron en responderme. El paquete traía el manual.
> **El material está por encima de la media**, se nota al tocarlo. Todavía no me
> decido si me quedo con él.  → etiqueta: **negativo**

Un modelo lineal sobre bolsa de palabras **suma** las contribuciones de todas las
cláusulas, así que estructuralmente no puede representar "la polaridad de la
cláusula que manda". Por eso se estanca cerca de 89 % por más que se afine el
clasificador.

El marcado E/P/L del envío 2 intentaba atacar esto dándole más peso a la última
cláusula, pero se desperdicia: la última cláusula del texto suele ser una muletilla
de cierre sin opinión (*"lo dejo a tu criterio"*, *"ni fu ni fa"*, *"cada quien que
juzgue"*).

## 2. La idea: numerar las cláusulas por recencia de OPINIÓN

En lugar de marcar las cláusulas por su posición en el texto (E/P/L), se marcan por
su posición **entre las cláusulas que opinan**:

```
O1 = última cláusula de opinión (la candidata a decisiva)
O2 = penúltima cláusula de opinión
O3 = las anteriores
F  = cláusula de relleno (logística o muletilla de cierre)
```

Para eso hay que resolver tres problemas, y los tres se resuelven **aprendiendo de
los datos**, sin escribir listas de palabras a mano:

**a) ¿Qué cláusulas opinan?** Las reseñas neutrales del dataset solo hablan de
logística: nunca opinan. Sirven entonces como ejemplos negativos limpios. Los
positivos se obtienen de un detector por vocabulario (una palabra que **nunca**
aparece en reseñas neutrales es vocabulario de opinión), y con esos dos conjuntos se
entrena un clasificador de cláusulas.

El paso del detector por vocabulario al detector aprendido fue el que más aportó. El
detector por vocabulario tiene precisión alta pero se le escapan las opiniones
frasales armadas con palabras que también usan las neutrales: *"cumple de sobra"*,
*"no vale lo que cuesta"*, *"se queda corta"*, *"por encima de la media"*. El
clasificador generaliza a esas porque aprende la **sintaxis evaluativa**
(`<aspecto> <verbo evaluativo> <valoración>`) y no solo el léxico.

| Detector | Acierto de la polaridad por cláusula |
| --- | --- |
| por vocabulario | 84,5 % |
| **aprendido** | **97,8 %** |

**b) ¿Qué polaridad tiene cada cláusula?** Las reseñas con **una sola** cláusula de
opinión dan esa polaridad sin ambigüedad (~1.900 ejemplos). Para las demás se usa su
última cláusula de opinión, que coincide con la etiqueta de la reseña en el 91 % de
los casos.

Lo que **no** funciona es etiquetar todas las cláusulas con la etiqueta de su reseña
(supervisión distante plana): una misma cláusula aparece como decisiva en unas
reseñas y como distractora en otras, su etiqueta queda ~50/50 y el modelo resultante
llega apenas a 61 %.

**c) ¿Cuál opinión manda?** Tomando las reseñas con dos cláusulas de opinión de
polaridad opuesta y comparando con la etiqueta real:

| Situación | Gana la opinión posterior |
| --- | --- |
| en general | 91 % |
| la posterior lleva marcador de resumen (*al final*, *con todo*, *a fin de cuentas*) | 97 % |
| la posterior lleva marcador de concesión (*eso sí*, *aun así*) | 93 % |
| la posterior lleva marcador secundario (*de paso*, *por cierto*) | **39 %** |

Es decir, *de paso* y *por cierto* señalan una opinión que **no** decide. Estas
reglas no se codifican a mano: los marcadores se emiten como tokens junto al rol de
su cláusula (`O1_MARC_resumen`, `O2_MARC_secundario`, ...) y el clasificador final
aprende a pesarlas.

## 3. Importaciones y datos

La lógica de cláusulas vive en el módulo `features_grupo18.py`, que está junto a este
notebook. Está en un archivo aparte y no dentro del notebook **a propósito**: el
enunciado pide entregar el modelo serializado con `joblib`, y un modelo que contiene
clases definidas en el notebook no se puede volver a cargar en otra máquina.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import joblib
import matplotlib.pyplot as plt

import nltk
nltk.download('stopwords', quiet=True)

# Módulo propio del grupo: separación en cláusulas, detector de opinión y rasgos.
# Vive en un .py aparte para que el modelo se pueda guardar y recargar con joblib.
sys.path.insert(0, os.getcwd())      # para que el módulo se encuentre junto al notebook
import features_grupo18 as F

df_train = pd.read_csv("train.csv")
y_train = df_train['label']
print(f"{len(df_train)} reseñas | clases: {y_train.value_counts().to_dict()}")

### Evidencia del diagnóstico

Se reproduce el hallazgo que motiva todo el envío: las palabras de opinión están
repartidas entre las dos polaridades, así que su presencia no indica la clase.

In [ ]:
from collections import Counter

# El detector aprende qué palabras son de opinión a partir de las reseñas neutrales
detector = F.DetectorOpinion().fit(df_train['text'], y_train)
print(f"clase neutral inferida de los datos: {detector.clase_neutral_}")
print(f"vocabulario de opinión (semilla): {len(detector.vocab_opinion_)} palabras\n")

# ¿En qué clase aparece cada palabra de opinión?
cuenta = {}
for texto, etiqueta in zip(df_train['text'], y_train):
    for clausula in F.separar_clausulas(texto):
        for palabra in set(F.tokens_crudos(clausula)):
            if palabra in detector.vocab_opinion_:
                cuenta.setdefault(palabra, Counter())[etiqueta] += 1

filas = []
for palabra, c in cuenta.items():
    p, n = c['positivo'], c['negativo']
    if p + n >= 100:
        filas.append({'palabra': palabra, 'en_positivas': p, 'en_negativas': n,
                      'sesgo': (p - n) / (p + n)})
tabla = pd.DataFrame(filas).sort_values('sesgo')
print("Palabras de opinión frecuentes, ordenadas por sesgo hacia una clase:")
print(pd.concat([tabla.head(6), tabla.tail(6)]).to_string(index=False,
      float_format=lambda v: f"{v:+.2f}"))
print(f"\nsesgo medio absoluto: {tabla['sesgo'].abs().mean():.3f} "
      f"(0 = la palabra no dice nada sobre la clase)")

### Cómo queda marcada una reseña

Se compara el marcado por posición del envío 2 con el marcado por rol de opinión de
este envío, sobre una reseña que termina con una muletilla sin opinión.

In [ ]:
analizador_demo = F.AnalizadorDeResenas().fit(df_train['text'], y_train)

# Una reseña cuya última cláusula es relleno: ahí el marcado E/P/L se desperdicia
for texto, etiqueta in zip(df_train['text'], y_train):
    clausulas, roles = analizador_demo.det_.analizar_lote([texto])[0]
    if len(clausulas) >= 4 and roles[-1] == 'F' and sum(r.startswith('O') for r in roles) >= 2:
        break

print("TEXTO (etiqueta real:", etiqueta, ")\n")
print(texto, "\n")
print(f"{'rol':>4s}  {'envío 2':>8s}   cláusula")
n = len(clausulas)
for i, (clausula, rol) in enumerate(zip(clausulas, roles)):
    epl = "L" if i == n - 1 else ("P" if i == n - 2 else "E")
    marcas = F.marcadores_de(clausula)
    extra = f"   {marcas}" if marcas else ""
    print(f"{rol:>4s}  {epl:>8s}   {clausula[:78]}{extra}")

## 4. Representación

El analizador produce, de una sola pasada sobre el texto crudo, varias vistas de
cada reseña. Cada vista alimenta su propio vectorizador y todo se concatena con
`ColumnTransformer`:

| Bloque | Vista | Vectorizador | Para qué |
| --- | --- | --- | --- |
| `epl` | marcado por posición E/P/L | TF-IDF palabras (1,2) | se conserva del envío 2, sigue aportando |
| `roles` | marcado por rol O1/O2/O3/F + marcadores | TF-IDF palabras (1,2) | el bloque nuevo: *cuál* opinión manda |
| `car_ult` | última cláusula del texto | TF-IDF caracteres `char_wb` (2,5) | robustez a erratas, del envío 2 |
| `car_o1` | última cláusula de **opinión** | TF-IDF caracteres `char_wb` (2,5) | erratas en la cláusula decisiva |
| `pal_o1` | última cláusula de **opinión** | TF-IDF palabras (1,3) | expresiones frasales completas |

Los dos bloques del envío 2 se mantienen porque se midió que **sumaban**: quitarlos
y dejar solo los bloques nuevos daba 88,67 %, por debajo del envío 3.

Todo va dentro de un único `Pipeline`, así que el detector, el modelo de polaridad y
los vocabularios TF-IDF se ajustan **solo con el fold de entrenamiento** en cada
partición de la validación cruzada. No hay fuga de información.

### Cómo funciona el analizador por dentro

`AnalizadorDeResenas` es un transformador de scikit-learn definido en `features_grupo18.py`. Al
ajustarlo (`fit`) con las reseñas de entrenamiento y sus etiquetas, hace lo siguiente:

1. **Identifica la clase neutral** a partir de los datos, sin depender de su nombre. Para cada clase
   cuenta cuántas palabras frecuentes de train no aparecen nunca en sus reseñas; la neutral es la que
   tiene más ausencias, porque le falta el vocabulario de opinión de las dos polaridades, mientras que
   a las positivas solo les falta el negativo y viceversa. Se hace así porque algunos meta-estimadores
   de scikit-learn (como `StackingClassifier`) convierten las etiquetas en enteros y no se puede buscar
   la palabra «neutral».
2. **Arma el vocabulario de opinión:** las palabras que aparecen al menos 5 veces en train y ninguna
   vez en una reseña neutral.
3. **Entrena el detector de opinión**, un `LinearSVC` sobre TF-IDF de palabras (1 a 3) y de caracteres
   (2 a 5) que clasifica cláusulas. Los ejemplos negativos son todas las cláusulas de las reseñas
   neutrales; los positivos, las cláusulas de reseñas con opinión que contienen alguna palabra del
   vocabulario del paso 2. Se usa `class_weight='balanced'` porque los dos grupos no tienen el mismo
   tamaño.
4. **Entrena un modelo de polaridad por cláusula** con la última cláusula de opinión de cada reseña y
   la etiqueta de la reseña (la pseudo-etiqueta del 91 % descrita en la sección 2).

Al transformar (`transform`), el analizador separa cada reseña en cláusulas, puntúa todas con el
detector y marca como opinión las que superan el umbral (0 por defecto). Recorriendo las cláusulas de
opinión desde el final, asigna `O1` a la última, `O2` a la penúltima y `O3` al resto; las que no
opinan quedan como `F`. Con esos roles construye las vistas de texto de la tabla anterior.

**Qué se usa y qué no en el modelo final.** El analizador también produce una quinta vista de texto
(`o1_o2`, las dos últimas cláusulas de opinión) y 38 rasgos numéricos (polaridad por rol, conteos de
cláusulas, marcadores de discurso), que son los que usa el modelo de polaridad del paso 4. El modelo
final **no los usa**: el `ColumnTransformer` solo toma las cuatro columnas de texto de la tabla y
descarta el resto. Esos rasgos se conservan en el módulo porque fueron la base de varios experimentos
de la sección 6 (por ejemplo, HistGradientBoosting y stacking), donde se midió que no superaban al
modelo de solo texto. Por lo mismo, el modelo de polaridad se entrena dentro del pipeline pero no
influye en las predicciones.

In [ ]:
def tfidf_marcado(**kw):
    """TF-IDF de palabras sobre tokens que llevan prefijo de rol o de posición."""
    base = dict(
        ngram_range=(1, 2),
        sublinear_tf=True,       # suaviza palabras repetidas: 1 + log(tf)
        min_df=2,
        token_pattern=r'\S+',    # cada token cuenta entero, con su prefijo
        lowercase=False,         # conserva los prefijos O1_, O2_, F_, E_, P_, L_
    )
    base.update(kw)
    return TfidfVectorizer(**base)


def tfidf_caracteres():
    """n-gramas de caracteres dentro de los límites de cada palabra."""
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3,
                           sublinear_tf=True)


# Solo se usan cuatro columnas de texto del analizador. Las demás (o1_o2 y los 38 rasgos
# numéricos) se descartan, porque ColumnTransformer usa remainder='drop' por defecto.
representacion = ColumnTransformer([
    ('epl',      tfidf_marcado(),                                      'epl'),
    ('roles',    tfidf_marcado(),                                      'roles'),
    ('car_ult',  tfidf_caracteres(),                                   'ultima'),
    ('car_o1',   tfidf_caracteres(),                                   'o1'),
    ('pal_o1',   TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=2), 'o1'),
])


def construir_modelo():
    """Pipeline completo: texto crudo -> vistas -> TF-IDF -> LinearSVC."""
    return Pipeline([
        ('analizador', F.AnalizadorDeResenas()),
        ('representacion', representacion),
        ('clf', LinearSVC(
            C=0.5,           # inverso de la regularización, elegido por CV
            random_state=42,
        )),
    ])


modelo = construir_modelo()
modelo

## 5. Evaluación

Se usa la **misma** validación cruzada de los envíos 2 y 3 (`StratifiedKFold`, 5
particiones, `shuffle=True`, `random_state=42`), de modo que las tres cifras son
comparables partición por partición.

El clasificador final es el mismo `LinearSVC` con `C=0.5` del envío 3, a propósito: así la mejora
que se mida se puede atribuir al cambio de representación y no a un cambio de clasificador. El envío 3
explica en detalle cómo funciona el LinearSVC y por qué se eligió ese valor de `C`; en este envío el
`C` se volvió a barrer junto con los demás hiperparámetros (ver la sección 6) y `0.5` siguió estando
dentro de la meseta de mejores resultados.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(modelo, df_train['text'], y_train, cv=cv, scoring='accuracy')
print("Accuracy por fold:", scores.round(4))
print(f"Accuracy promedio: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Las cifras de los envíos 2 y 3 se copian de sus notebooks (misma CV, mismos folds)
comparacion = pd.DataFrame({
    'Envío': ['2 - Reg. Logística C=10', '3 - LinearSVC C=0.5',
              '4 - rol de opinión + LinearSVC C=0.5'],
    'Representación': ['posición E/P/L', 'posición E/P/L', 'rol de opinión O1/O2/O3/F'],
    'Accuracy CV': [0.8915, 0.8939, scores.mean()],
})
comparacion['Mejora vs envío 2'] = (comparacion['Accuracy CV'] - 0.8915) * 100
print()
print(comparacion.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Resultados por clase

Igual que en los envíos anteriores, el accuracy promedio no dice en qué clases se equivoca el modelo.
Con `cross_val_predict` se obtiene, para cada una de las 12.000 reseñas, la predicción del modelo del
fold en el que esa reseña quedó como validación. Con esas predicciones se construyen el reporte por
clase (precisión, recall y F1), la matriz de confusión y el conteo de cuántos errores son confusiones
entre positivo y negativo, que es el número que este envío busca bajar.

In [ ]:
y_pred_cv = cross_val_predict(modelo, df_train['text'], y_train, cv=cv)

print(classification_report(y_train, y_pred_cv, digits=4))

ConfusionMatrixDisplay.from_predictions(y_train, y_pred_cv, cmap='Blues')
plt.title("Matriz de confusión - Envío 4 (CV 5 folds)")
plt.show()

errores = (y_train != y_pred_cv)
opinion = (y_train.isin(['positivo', 'negativo'])
           & pd.Series(y_pred_cv, index=y_train.index).isin(['positivo', 'negativo']))
print(f"Errores totales: {errores.sum()} | "
      f"confusiones positivo<->negativo: {(errores & opinion).sum()}")

### Conclusiones

El modelo alcanza **90,52 %** de accuracy en validación cruzada (±0,54 % entre
folds), frente a 89,39 % del envío 3 y 89,15 % del envío 2.

| Clase | F1 envío 2 | F1 envío 3 | F1 envío 4 |
| --- | --- | --- | --- |
| Neutral | 98,37 % | 98,11 % | **99,68 %** |
| Negativo | 85,33 % | 85,80 % | **86,66 %** |
| Positivo | 85,04 % | 85,41 % | **86,62 %** |

**La clase neutral queda resuelta.** Pasa de F1 98,1 % a 99,68 %: solo 23 de las
3.576 reseñas neutrales se confunden. Tiene una explicación directa y es la
confirmación de que el detector funciona: una reseña neutral es, por definición, una
reseña **sin ninguna cláusula de opinión**, y eso es exactamente lo que el detector
mide. Antes el modelo tenía que inferirlo del vocabulario; ahora lo lee de la
estructura.

**La mejora en positivo y negativo es real pero modesta** (+1,3 y +1,6 puntos de F1).
Las confusiones positivo ↔ negativo bajan de 1.136 a 1.114.

**El cuello de botella sigue siendo el mismo, y ahora es el único.** De los 1.137
errores que quedan, **1.114 (98 %) son confusiones positivo ↔ negativo**. Ya no hay
nada que ganar en neutral. Lo que falta es decidir correctamente *cuál* de las
opiniones de la reseña manda, y ahí el techo medido es claro:

- la polaridad por cláusula ya se acierta al **97,8 %**, y subirla a 98,9 % (es
  posible, quitando las pseudo-etiquetas) **no mejora el accuracy final**;
- la regla "la última opinión manda", aplicada sola, acierta solo **86,4 %** en las
  reseñas de opinión.

Es decir, el límite ya no está ni en detectar opiniones ni en puntuarlas, sino en la
regla de agregación: el 9 % de las veces la opinión decisiva **no** es la última, y
los marcadores de discurso no alcanzan para distinguir esos casos. Resolverlo
requiere modelar el orden de las cláusulas como una secuencia, que es precisamente
lo que habilita la Parte 2 del proyecto.

## 6. Qué se probó y se descartó

Esta sección documenta los caminos que se midieron y **no** funcionaron, porque
delimitan el techo del enfoque clásico y justifican por qué el modelo final es el que
es. Todas las cifras son de un conjunto de validación separado (25 % de train).

| Enfoque probado | Resultado | Por qué se descartó |
| --- | --- | --- |
| Supervisión distante plana: etiquetar **todas** las cláusulas con la etiqueta de su reseña | 0,61 | Una misma cláusula es decisiva en unas reseñas y distractora en otras: su etiqueta queda ~50/50 |
| Clasificar por regla: "la última opinión manda", con polaridad por cláusula | 0,870 | Peor que el modelo de texto; la regla falla en el 9 % de los casos |
| Marcado por rol con detector **por vocabulario** | 0,887 | El detector se pierde las opiniones frasales y contamina la supervisión |
| Solo los 38 rasgos de estructura de opinión + HistGradientBoosting | 0,890 | Señal real pero más débil que el TF-IDF; útil solo como evidencia |
| Concatenar los rasgos densos al TF-IDF en el mismo modelo lineal | 0,869 | Los valores densos aplastan al TF-IDF disperso; con peso 0,05 sube a 0,901, aún por debajo |
| `StackingClassifier` (texto + rasgos) | 0,893 | El meta-modelo no logra decidir cuándo confiar en cada base |
| Stacking dándole al meta-modelo los rasgos como contexto | 0,880 | Sobreajusta el desfase entre predicciones fuera de fold y del modelo completo |
| Auto-entrenamiento del modelo de polaridad (4 rondas) | se estanca | La polaridad por cláusula no se mueve de 0,845 con el detector por vocabulario |
| Ensamble por votación de 3 clasificadores lineales | 0,906 | Empate con el mejor modelo solo: los lineales coinciden en el 98 % de las predicciones |

Dato que explica por qué los ensambles no ayudan: el modelo de texto y el de rasgos
coinciden en el 92,3 % de sus predicciones y la "unión potencial" (acierta al menos
uno) es 0,935; pero ningún meta-modelo probado logró acercarse a ese número, porque
la información que diría a cuál creerle es justamente la que falta.

**Hiperparámetros.** Se barrieron tanto los del clasificador final (`LinearSVC` con
C entre 0,25 y 2, `LogisticRegression`, `RidgeClassifier`, `SGDClassifier`,
`class_weight`) como los del analizador (umbral del detector, C del detector, C del
modelo de polaridad, frecuencia mínima del vocabulario, uso de pseudo-etiquetas) y
los rangos de n-gramas. Todas las combinaciones razonables caen entre **0,9037 y
0,9052** en validación cruzada: el resultado es una meseta, no un pico, así que se
eligió la configuración más simple. Un `RandomizedSearchCV` sobre este espacio no
tiene margen que recuperar.

## 7. Generación de las predicciones y del archivo del modelo

Se reentrena el pipeline completo con el 100 % de train y se predice sobre el texto
crudo de `eval`. El modelo se guarda con `joblib`, como pide el enunciado, y se
verifica que al recargarlo produce las mismas predicciones.

In [ ]:
# Reentrenar con el 100% de los datos de train
modelo.fit(df_train['text'], y_train)

df_eval = pd.read_csv("eval.csv")
df_sub = pd.read_csv("sample_submission.csv")
df_sub['answer'] = modelo.predict(df_eval['text'])

print("Distribución de predicciones (proporción):")
print(df_sub['answer'].value_counts(normalize=True).round(4))

sub_v3 = pd.read_csv("submission_linearsvc_v3_grupo18.csv")
cambios = int((sub_v3['answer'].values != df_sub['answer'].values).sum())
print(f"\nFilas: {len(df_sub)} | predicciones distintas al envío 3: "
      f"{cambios} ({cambios / len(df_sub):.1%})")

df_sub.to_csv("submission_opinion_v4_grupo18.csv", index=False)
print("Archivo generado: submission_opinion_v4_grupo18.csv")

In [ ]:
# Guardado del modelo entrenado, como exige el enunciado
joblib.dump(modelo, "modelo_final_grupo18.joblib", compress=3)

# Verificación de que el modelo se puede volver a cargar y predice lo mismo.
# Funciona porque las clases propias están en features_grupo18.py y no en el notebook.
recargado = joblib.load("modelo_final_grupo18.joblib")
iguales = (recargado.predict(df_eval['text'][:300]) == df_sub['answer'].values[:300]).all()
print(f"modelo_final_grupo18.joblib recargado | predicciones idénticas: {iguales}")

### Nota sobre cómo elegir el envío

La elección entre los envíos debe hacerse con la **validación cruzada**, no con el
score público del leaderboard. El público se calcula sobre una fracción de los datos
de test y una diferencia de unas décimas entra dentro de su ruido; la CV usa las
12.000 reseñas de train y aquí la ventaja del envío 4 es de +1,13 puntos sobre el
envío 3, consistente entre folds.

Para la entrega en Bloque Neón corresponde subir este notebook junto con
`features_grupo18.py` y `modelo_final_grupo18.joblib`. El módulo es necesario para
recargar el modelo: sin él, `joblib.load` no puede reconstruir las clases propias.